# Setting up / importing

In [ ]:
import sys

PROJECT_DIR = "/home/565/pv3484/aus_substation_electricity"
sys.path.append(PROJECT_DIR)
%cd {PROJECT_DIR}

In [ ]:
%run /home/565/pv3484/aus_substation_electricity/import_substation.py

In [ ]:
import time

import pandas as pd
from dateutil.easter import easter
from geopy.geocoders import Nominatim

RANK_CSV = "/home/565/pv3484/aus_substation_electricity/data/cleaned_data/full_nsw_relative_rank.csv"


def second_monday_of_june(y):
    """Return the date of the second Monday in June for year y (Monarch's Birthday)."""
    june = pd.date_range(start=f"{y}-06-01", end=f"{y}-06-30", freq="D")
    mondays = june[june.weekday == 0]
    return mondays[1]


# NSW public holidays, including Easter (date-calculated each year)
HOLIDAYS_VIC = {
    "New Year's Day":  lambda y: pd.Timestamp(f"{y}-01-01"),
    "Australia Day":   lambda y: pd.Timestamp(f"{y}-01-26"),
    "Good Friday":     lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=2),
    "Easter Saturday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=1),
    "Easter Sunday":   lambda y: pd.Timestamp(easter(y)),
    "Easter Monday":   lambda y: pd.Timestamp(easter(y)) + pd.Timedelta(days=1),
    "ANZAC Day":       lambda y: pd.Timestamp(f"{y}-04-25"),
    "Monarch's Birthday": lambda y: second_monday_of_june(y),
    "Christmas Day":   lambda y: pd.Timestamp(f"{y}-12-25"),
    "Boxing Day":      lambda y: pd.Timestamp(f"{y}-12-26"),
}

holiday_order = list(HOLIDAYS_VIC.keys())

# Maps individual holiday names to their group label
HOLIDAY_GROUPS = {
    "Good Friday":     "Easter Long Weekend",
    "Easter Saturday": "Easter Long Weekend",
    "Easter Sunday":   "Easter Long Weekend",
    "Easter Monday":   "Easter Long Weekend",
    "Christmas Day":   "Christmas and Boxing Day",
    "Boxing Day":      "Christmas and Boxing Day",
}

In [ ]:
info.loc[info["Name"] == "Chatswood", "Residential"]

In [ ]:
rank = pd.read_csv(RANK_CSV)

In [ ]:
rank.columns.tolist()

## Geocode substations

Look up lat/lon for each substation by suburb name. Results are written back into `info`.
Manually patches **Dee Why West**, which has no polygon in the geocoder.

In [ ]:
geolocator = Nominatim(user_agent="sydney_demand_mapper")


def get_coords(place):
    """Return (lat, lon) for a NSW suburb name, or (None, None) if not found."""
    try:
        loc = geolocator.geocode(f"{place}, New South Wales, Australia")
        if loc:
            return loc.latitude, loc.longitude
    except Exception as e:
        print(f"Geocoding failed for {place}: {e}")
    return None, None


latitudes, longitudes = [], []
for suburb in info["Name"]:
    lat, lon = get_coords(suburb)
    latitudes.append(lat)
    longitudes.append(lon)
    time.sleep(1)  # respect Nominatim rate limit

info["latitude"] = latitudes
info["longitude"] = longitudes

In [ ]:
# Check which stations failed to geocode
missing = info[info["latitude"].isna() | info["longitude"].isna()]
missing["Name"].unique()

In [ ]:
# Dee Why West has no suburb polygon — patch with manually sourced coordinates
info.loc[info["Name"] == "Dee Why West", "latitude"] = -33.73441
info.loc[info["Name"] == "Dee Why West", "longitude"] = 151.28278

## Map mean relative rank differences

Three-panel map showing, for each high-residential substation:
- **Panel 1** – Public holiday mean relative rank (binned, plasma palette)
- **Panel 2** – PH minus weekend (separate scale, RdBu_r)
- **Panel 3** – PH minus weekday (separate scale, PuOr_r)

In [ ]:
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
import contextily as ctx
import numpy as np
import pathlib
import pandas as pd
from matplotlib.colors import ListedColormap

BLOCK_LABELS = {
    "00_04": "00:00–04:00",
    "04_10": "04:00–10:00",
    "10_15": "10:00–15:00",
    "15_20": "15:00–20:00",
    "20_24": "20:00–00:00",
}

ALL_BLOCKS = list(BLOCK_LABELS.keys())

HOLIDAY_BATCH = [
    ("New Year's Day",           ["New Year's Day"]),
    ("Australia Day",            ["Australia Day"]),
    ("Easter Long Weekend",      ["Good Friday", "Easter Saturday", "Easter Sunday", "Easter Monday"]),
    ("ANZAC Day",                ["ANZAC Day"]),
    ("Queen's Birthday",         ["Monarch's Birthday"]),
    ("Christmas and Boxing Day", ["Christmas Day", "Boxing Day"]),
]

FIG_ROOT = pathlib.Path(
    "/home/565/pv3484/aus_substation_electricity/data/figures"
    "/mrr_pb_minus_we_wd/separate_scale_new_scale"
)


def map_ph_we_wd_block_combined(
    df,
    info,
    holidays,
    holiday_group_name,
    block,
    we_diff_min=None,
    we_diff_max=None,
    wd_diff_min=None,
    wd_diff_max=None,
    df_station_col="Name",
    info_station_col="Name",
    lat_col="latitude",
    lon_col="longitude",
    residential_col="Residential",
    residential_threshold=0.75,
    block_suffix="_mean_relative_rank",
    crs_epsg=3857,
    wd_diff_cmap="PuOr_r",
):
    """Three-panel map of public holiday relative demand vs. weekends and weekdays.

    Parameters
    ----------
    df : DataFrame
        Relative rank data (one row per station-date).
    info : DataFrame
        Substation metadata including lat/lon and land-use fractions.
    holidays : list[str]
        Holiday names to treat as the public holiday group.
    holiday_group_name : str
        Display name for the holiday group (used in the title).
    block : str
        Time block key, e.g. ``"10_15"``.
    we_diff_min, we_diff_max : float, optional
        Colour scale limits for the PH−weekend panel. Auto-symmetrised if omitted.
    wd_diff_min, wd_diff_max : float, optional
        Colour scale limits for the PH−weekday panel. Auto-symmetrised if omitted.
    """
    block_label = BLOCK_LABELS.get(block, block)

    # --- Eligible stations: high-residential, with data, excluding Umina ---
    high_res = info[
        (info[residential_col] >= residential_threshold)
        & (info[info_station_col] != "Umina")
    ][info_station_col]

    block_cols = [f"{b}{block_suffix}" for b in ALL_BLOCKS]
    stations_with_data = (
        df.groupby(df_station_col)[block_cols]
        .apply(lambda x: x.notna().any().any())
    )
    stations_with_data = stations_with_data[stations_with_data].index
    eligible_stations = set(high_res).intersection(stations_with_data)

    df = df[df[df_station_col].isin(eligible_stations)].copy()
    df_hol = df[df["holiday_group"].isin(holidays)].copy()

    # --- GeoDataFrame ---
    gdf = info[info[info_station_col].isin(eligible_stations)].copy()
    gdf = gpd.GeoDataFrame(
        gdf,
        geometry=gpd.points_from_xy(gdf[lon_col], gdf[lat_col]),
        crs="EPSG:4326",
    ).to_crs(crs_epsg)

    # --- Compute PH, PH−WE, PH−WD per station ---
    col = f"{block}{block_suffix}"
    results = []

    for station in eligible_stations:
        df_s = df_hol[df_hol[df_station_col] == station]

        if df_s.empty or df_s[col].isna().all():
            results.append({info_station_col: station, "ph": np.nan, "ph_we": np.nan, "ph_wd": np.nan})
            continue

        df_station_all = df[df[df_station_col] == station]

        ph_val = df_s[df_s["is_holiday"]][col].mean()
        we_val = df_station_all[~df_station_all["is_holiday"] & df_station_all["is_weekend"]][col].mean()
        wd_val = df_station_all[~df_station_all["is_holiday"] & ~df_station_all["is_weekend"]][col].mean()

        results.append({
            info_station_col: station,
            "ph":    ph_val,
            "ph_we": ph_val - we_val if pd.notna(ph_val) and pd.notna(we_val) else np.nan,
            "ph_wd": ph_val - wd_val if pd.notna(ph_val) and pd.notna(wd_val) else np.nan,
        })

    gdf = gdf.merge(pd.DataFrame(results), on=info_station_col, how="left")

    # Guard against empty data
    if gdf["ph"].isna().all():
        plt.close("all")
        raise ValueError(f"No data for holiday='{holiday_group_name}', block='{block}'")

    # --- PH bins and plasma palette ---
    bins = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
    bin_labels = [f"{bins[i]:.2f}–{bins[i+1]:.2f}" for i in range(len(bins) - 1)]

    gdf["ph_clipped"] = gdf["ph"].clip(lower=bins[0], upper=bins[-1])
    gdf["ph_bin"] = pd.cut(gdf["ph_clipped"], bins=bins, labels=bin_labels, include_lowest=True)

    plasma = matplotlib.colormaps.get_cmap("plasma")
    ph_colors = [plasma(p) for p in np.linspace(0, 1, len(bin_labels)) ** 0.7]
    ph_cmap = ListedColormap(ph_colors)

    # --- Auto-symmetrise difference scales if not provided ---
    if we_diff_min is None:
        abs_we = max(abs(gdf["ph_we"].min()), abs(gdf["ph_we"].max()))
        we_diff_min, we_diff_max = -abs_we, abs_we
    if wd_diff_min is None:
        abs_wd = max(abs(gdf["ph_wd"].min()), abs(gdf["ph_wd"].max()))
        wd_diff_min, wd_diff_max = -abs_wd, abs_wd

    # --- Map extent with padding ---
    minx, miny, maxx, maxy = gdf.total_bounds
    pad = 4800
    extent = (minx - pad, maxx + pad, miny - pad, maxy + pad)

    # --- Plotting ---
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    for ax in axes:
        ax.set_aspect("equal", adjustable="box")

    year_min = int(df_hol["year"].min()) if not df_hol.empty else "?"
    year_max = int(df_hol["year"].max()) if not df_hol.empty else "?"

    fig.suptitle(
        f"Relative Demand Patterns ({holiday_group_name}) "
        f"({block_label}, {year_min}–{year_max})\n"
        f"High-residential locations only (≥ {residential_threshold})",
        fontsize=17, y=0.999,
    )

    title_y = 0.99
    axes[0].set_title("Public Holiday Relative Rank (binned)", y=title_y)
    axes[1].set_title("Public Holiday − Weekend Mean Relative Rank", y=title_y)
    axes[2].set_title("Public Holiday − Weekday Mean Relative Rank", y=title_y)

    MARKER_KWARGS = dict(markersize=70, edgecolor="black", legend=False,
                         missing_kwds={"color": "#e6e6e6", "label": "No data"})

    # Panel 1: PH binned
    gdf.plot(ax=axes[0], column="ph_bin", cmap=ph_cmap, **MARKER_KWARGS)

    handles = [
        plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=ph_colors[i], markersize=10)
        for i in range(len(bin_labels))
    ]
    handles.append(
        plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#e6e6e6", markersize=10, label="No data")
    )
    fig.legend(
        handles, bin_labels + ["No data"],
        title="Public Holiday Relative Rank (binned)\n(values > 0.60 clipped to top bin)",
        loc="center left", bbox_to_anchor=(0.005, 0.50), frameon=False, fontsize=8,
    )

    # Panel 2: PH − Weekend
    gdf.plot(ax=axes[1], column="ph_we", cmap="RdBu_r", vmin=we_diff_min, vmax=we_diff_max, **MARKER_KWARGS)
    sm_we = plt.cm.ScalarMappable(cmap="RdBu_r", norm=plt.Normalize(vmin=we_diff_min, vmax=we_diff_max))
    sm_we._A = []
    fig.colorbar(sm_we, ax=axes[1], orientation="horizontal", fraction=0.05, pad=0.02).set_label(
        f"PH − Weekend ({we_diff_min:.2f} to {we_diff_max:.2f})"
    )

    # Panel 3: PH − Weekday
    gdf.plot(ax=axes[2], column="ph_wd", cmap=wd_diff_cmap, vmin=wd_diff_min, vmax=wd_diff_max, **MARKER_KWARGS)
    sm_wd = plt.cm.ScalarMappable(cmap=wd_diff_cmap, norm=plt.Normalize(vmin=wd_diff_min, vmax=wd_diff_max))
    sm_wd._A = []
    fig.colorbar(sm_wd, ax=axes[2], orientation="horizontal", fraction=0.05, pad=0.02).set_label(
        f"PH − Weekday ({wd_diff_min:.2f} to {wd_diff_max:.2f})"
    )

    # Basemap and station labels
    for ax in axes:
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
        for _, row in gdf.iterrows():
            ax.text(row.geometry.x + 800, row.geometry.y, row[info_station_col],
                    fontsize=8, ha="left", va="center")
        ax.set_axis_off()

    plt.subplots_adjust(top=0.86, bottom=0.16, wspace=0.08, hspace=0.01)
    return fig


def compute_global_diff_scales(df, holiday_batch, block_suffix="_mean_relative_rank",
                                df_station_col="Name"):
    """Compute symmetric colour scale limits for PH−WE and PH−WD across all holidays and blocks."""
    all_we, all_wd = [], []

    for _, holidays in holiday_batch:
        df_hol = df[df["holiday_group"].isin(holidays)]

        for block in ALL_BLOCKS:
            col = f"{block}{block_suffix}"

            for station in df[df_station_col].unique():
                df_s = df_hol[df_hol[df_station_col] == station]
                df_all = df[df[df_station_col] == station]

                ph_val = df_s[df_s["is_holiday"]][col].mean() if not df_s.empty else np.nan
                we_val = df_all[~df_all["is_holiday"] & df_all["is_weekend"]][col].mean()
                wd_val = df_all[~df_all["is_holiday"] & ~df_all["is_weekend"]][col].mean()

                if pd.notna(ph_val) and pd.notna(we_val):
                    all_we.append(ph_val - we_val)
                if pd.notna(ph_val) and pd.notna(wd_val):
                    all_wd.append(ph_val - wd_val)

    abs_we = max(abs(min(all_we)), abs(max(all_we)))
    abs_wd = max(abs(min(all_wd)), abs(max(all_wd)))

    return (-abs_we, abs_we), (-abs_wd, abs_wd)


In [ ]:
fig = map_ph_we_wd_block_combined(
    df=rank,
    info=info,
    holidays=["Easter Long Weekend"],
    holiday_group_name="Easter Long Weekend",
    block="10_15",
)
plt.show()

## looping and saving

In [ ]:
def batch_export_mrr_maps(df, info, holiday_batch=HOLIDAY_BATCH, fig_root=FIG_ROOT):
    """Generate and save one PNG per holiday group × time block.

    Output layout::

        <fig_root>/<Holiday Group Name>/<block>.png
    """
    (we_min, we_max), (wd_min, wd_max) = compute_global_diff_scales(df, holiday_batch)
    print(f"Global scales — PH−WE: [{we_min:.3f}, {we_max:.3f}]  PH−WD: [{wd_min:.3f}, {wd_max:.3f}]")

    total = len(holiday_batch) * len(ALL_BLOCKS)
    done = 0

    for group_name, holidays in holiday_batch:
        folder_name = group_name.replace("/", "-").replace("'", "")
        out_dir = fig_root / folder_name
        out_dir.mkdir(parents=True, exist_ok=True)

        for block in ALL_BLOCKS:
            out_path = out_dir / f"{block}.png"
            try:
                fig = map_ph_we_wd_block_combined(
                    df=df,
                    info=info,
                    holidays=holidays,
                    holiday_group_name=group_name,
                    block=block,
                    we_diff_min=we_min,
                    we_diff_max=we_max,
                    wd_diff_min=wd_min,
                    wd_diff_max=wd_max,
                )
            except ValueError as e:
                print(f"  [SKIP] {e}")
                done += 1
                continue

            fig.savefig(out_path, dpi=150, bbox_inches="tight")
            plt.close(fig)
            done += 1
            print(f"[{done}/{total}] Saved: {out_path}")

    print("\nDone.")

In [ ]:
# batch_export_mrr_maps(df=rank, info=info)

# Using discrete bins for panels 2 and 3 rather than continuous scale bars

In [ ]:
from matplotlib.colors import BoundaryNorm

FIG_ROOT_SEMINAR = pathlib.Path(
    "/home/565/pv3484/aus_substation_electricity/data/figures/seminar_difference_maps"
)

def map_ph_we_wd_block_combined_discrete(
    df,
    info,
    holidays,
    holiday_group_name,
    block,
    df_station_col="Name",
    info_station_col="Name",
    lat_col="latitude",
    lon_col="longitude",
    residential_col="Residential",
    residential_threshold=0.75,
    block_suffix="_mean_relative_rank",
    crs_epsg=3857,
):
    """Three-panel map with fixed discrete colour bars on panels 2 and 3.

    Panel 1 : Public Holiday Relative Rank (binned plasma, vertical colourbar)
    Panel 2 : PH minus Weekend mean  (RdBu_r, fixed +/-0.40, 0.10 bins)
    Panel 3 : PH minus Weekday mean  (PuOr_r, fixed +/-0.60, 0.10 bins)

    Each colourbar carries a vertical label describing its metric.
    Figure title shows the holiday group name and formatted time block.
    Stations listed in highlight_stations are drawn with a bold ring.

    Returns
    -------
    fig : matplotlib.figure.Figure
    gdf : GeoDataFrame with computed ph / ph_we / ph_wd columns
    """
    # ---- Automatic holiday-group expansion ----
    if isinstance(holidays, str):
        holidays = [h for h, g in HOLIDAY_GROUPS.items() if g == holidays]

    expanded = []
    for h in holidays:
        if h in HOLIDAY_GROUPS.values():
            expanded.extend([hh for hh, g in HOLIDAY_GROUPS.items() if g == h])
        else:
            expanded.append(h)
    holidays = expanded

    # ---- Rename Monarch's Birthday -> Queen's Birthday in data ----
    df = df.copy()
    df["holiday_group"] = df["holiday_group"].replace(
        {"Monarch's Birthday": "Queen's Birthday"}
    )

    # ---- Fixed global colour-scale ranges ----
    WE_MIN, WE_MAX, WE_STEP = -0.40, 0.40, 0.10
    WD_MIN, WD_MAX, WD_STEP = -0.60, 0.60, 0.10

    # ---- Compute PH / PH-WE / PH-WD BEFORE filtering stations ----
    block_cols = [f"{b}{block_suffix}" for b in ALL_BLOCKS]
    col = f"{block}{block_suffix}"
    block_label = BLOCK_LABELS.get(block, block)

    df_hol_full = df[df["holiday_group"].isin(holidays)].copy()

    results = []
    for station in df[df_station_col].unique():
        df_s   = df_hol_full[df_hol_full[df_station_col] == station]
        df_all = df[df[df_station_col] == station]

        if df_s.empty or df_s[col].isna().all():
            results.append({info_station_col: station,
                             "ph": np.nan, "ph_we": np.nan, "ph_wd": np.nan})
            continue

        ph_val = df_s[df_s["is_holiday"]][col].mean()
        we_val = df_all[~df_all["is_holiday"] &  df_all["is_weekend"]][col].mean()
        wd_val = df_all[~df_all["is_holiday"] & ~df_all["is_weekend"]][col].mean()

        results.append({
            info_station_col: station,
            "ph":    ph_val,
            "ph_we": ph_val - we_val if pd.notna(ph_val) and pd.notna(we_val) else np.nan,
            "ph_wd": ph_val - wd_val if pd.notna(ph_val) and pd.notna(wd_val) else np.nan,
        })

    results_df = pd.DataFrame(results)

    # ---- Filter to high-residential stations with data ----
    high_res = info[
        (info[residential_col] >= residential_threshold)
        & (info[info_station_col] != "Umina")
    ][info_station_col]

    stations_with_data = (
        df.groupby(df_station_col)[block_cols]
          .apply(lambda x: x.notna().any().any())
    )
    stations_with_data = stations_with_data[stations_with_data].index
    eligible_stations  = set(high_res).intersection(stations_with_data)

    results_df = results_df[results_df[info_station_col].isin(eligible_stations)]
    gdf = info[info[info_station_col].isin(eligible_stations)].copy()
    gdf = gdf.merge(results_df, on=info_station_col, how="left")

    # ---- GeoDataFrame ----
    gdf = gpd.GeoDataFrame(
        gdf,
        geometry=gpd.points_from_xy(gdf[lon_col], gdf[lat_col]),
        crs="EPSG:4326",
    ).to_crs(crs_epsg)

    # ---- Panel 1: PH bins (plasma) ----
    bins_ph    = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
    bin_labels = [f"{bins_ph[i]:.2f}-{bins_ph[i+1]:.2f}" for i in range(len(bins_ph) - 1)]

    gdf["ph_clipped"] = gdf["ph"].clip(lower=bins_ph[0], upper=bins_ph[-1])
    gdf["ph_bin"] = pd.cut(gdf["ph_clipped"], bins=bins_ph,
                            labels=bin_labels, include_lowest=True)

    plasma    = matplotlib.colormaps.get_cmap("plasma")
    ph_colors = [plasma(p) for p in np.linspace(0, 1, len(bin_labels)) ** 0.7]
    ph_cmap   = ListedColormap(ph_colors)

    # ---- Discrete bins for panels 2 & 3 ----
    we_bounds  = np.round(np.arange(WE_MIN, WE_MAX + WE_STEP, WE_STEP), 10)
    wd_bounds  = np.round(np.arange(WD_MIN, WD_MAX + WD_STEP, WD_STEP), 10)

    we_base    = matplotlib.colormaps.get_cmap("RdBu_r")
    wd_base    = matplotlib.colormaps.get_cmap("PuOr_r")
    we_listed  = ListedColormap([we_base(i / (len(we_bounds) - 2)) for i in range(len(we_bounds) - 1)])
    wd_listed  = ListedColormap([wd_base(i / (len(wd_bounds) - 2)) for i in range(len(wd_bounds) - 1)])
    we_norm    = BoundaryNorm(we_bounds, len(we_bounds) - 1)
    wd_norm    = BoundaryNorm(wd_bounds, len(wd_bounds) - 1)

    # ---- Map extent ----
    minx, miny, maxx, maxy = gdf.total_bounds
    pad    = 4800
    extent = (minx - pad, maxx + pad, miny - pad, maxy + pad)

    # ---- Figure ----
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    for ax in axes:
        ax.set_aspect("equal", adjustable="box")

    # ---- Figure-level title: holiday group + time block + year range ----
    year_min = int(df_hol_full["year"].min()) if not df_hol_full.empty else "?"
    year_max = int(df_hol_full["year"].max()) if not df_hol_full.empty else "?"
    fig.suptitle(
        f"{holiday_group_name}  |  {block_label}  ({year_min}–{year_max})",
        fontsize=16, fontweight="bold", y=1.01,
    )

    axes[0].set_title("Public Holiday Relative Rank", y=0.99)
    axes[1].set_title("Difference from Weekend Mean", y=0.99)
    axes[2].set_title("Difference from Weekday Mean", y=0.99)

    MARKER_KWARGS = dict(
        markersize=70,
        edgecolor="black",
        legend=False,
        missing_kwds={"color": "#e6e6e6", "label": "No data"},
    )

    # Panel 1
    gdf.plot(ax=axes[0], column="ph_bin", cmap=ph_cmap, **MARKER_KWARGS)
    ph_norm = BoundaryNorm(bins_ph, len(bins_ph) - 1)
    sm_ph = plt.cm.ScalarMappable(cmap=ph_cmap, norm=ph_norm)
    sm_ph._A = []
    cb0 = fig.colorbar(sm_ph, ax=axes[0], orientation="vertical",
                       fraction=0.046, pad=0.02, location="left",
                       boundaries=bins_ph, ticks=bins_ph)
    cb0.set_label("Mean Relative Rank", fontsize=8)

    # Panel 2
    gdf.plot(ax=axes[1], column="ph_we", cmap=we_listed, norm=we_norm, **MARKER_KWARGS)
    sm_we = plt.cm.ScalarMappable(cmap=we_listed, norm=we_norm)
    sm_we._A = []
    cb1 = fig.colorbar(sm_we, ax=axes[1], orientation="vertical",
                       fraction=0.046, pad=0.02, location="left",
                       boundaries=we_bounds, ticks=we_bounds)
    cb1.set_label("Difference in Mean Relative Rank", fontsize=8)

    # Panel 3
    gdf.plot(ax=axes[2], column="ph_wd", cmap=wd_listed, norm=wd_norm, **MARKER_KWARGS)
    sm_wd = plt.cm.ScalarMappable(cmap=wd_listed, norm=wd_norm)
    sm_wd._A = []
    cb2 = fig.colorbar(sm_wd, ax=axes[2], orientation="vertical",
                       fraction=0.046, pad=0.02, location="left",
                       boundaries=wd_bounds, ticks=wd_bounds)
    cb2.set_label("Difference in Mean Relative Rank", fontsize=8)

    # ---- Basemap + labels ----
    for ax in axes:
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
        for _, row in gdf.iterrows():
            ax.text(row.geometry.x + 800, row.geometry.y, row[info_station_col],
                    fontsize=8, ha="left", va="center")
        ax.set_axis_off()

    plt.subplots_adjust(top=0.86, bottom=0.16, wspace=0.08, hspace=0.01)
    return fig, gdf


def batch_export_discrete_maps(
    df,
    info,
    holiday_batch=HOLIDAY_BATCH,
    fig_root=FIG_ROOT_SEMINAR,
    highlight_stations=("Punchbowl", "Dulwich Hill"),
):
    """Loop over all holiday groups x time blocks and save one PNG each.

    Output layout::

        <fig_root>/<Holiday Group Name>/<block>.png

    Parameters
    ----------
    df : DataFrame
        Relative rank data (one row per station-date).
    info : DataFrame
        Substation metadata including lat/lon and land-use fractions.
    holiday_batch : list[tuple]
        List of (group_name, [holiday_keys]) pairs. Defaults to HOLIDAY_BATCH.
    fig_root : Path
        Root output directory. A sub-folder is created per holiday group.
    highlight_stations : sequence[str]
        Stations to mark with a bold ring on every figure.
    """
    total = len(holiday_batch) * len(ALL_BLOCKS)
    done  = 0

    for group_name, holidays in holiday_batch:
        folder_name = group_name.replace("/", "-").replace("'", "")
        out_dir = fig_root / folder_name
        out_dir.mkdir(parents=True, exist_ok=True)

        for block in ALL_BLOCKS:
            out_path = out_dir / f"{block}.png"
            try:
                fig, _ = map_ph_we_wd_block_combined_discrete(
                    df=df,
                    info=info,
                    holidays=holidays,
                    holiday_group_name=group_name,
                    block=block,
                    highlight_stations=highlight_stations,
                )
            except Exception as e:
                print(f"  [SKIP] {group_name} / {block} — {e}")
                done += 1
                continue

            fig.savefig(out_path, dpi=150, bbox_inches="tight")
            plt.close(fig)
            done += 1
            print(f"[{done}/{total}] Saved: {out_path}")

    print("\nDone.")

In [ ]:
fig, gdf = map_ph_we_wd_block_combined_discrete(
    df=rank,
    info=info,
    holidays=["Christmas and Boxing Day"],
    holiday_group_name="Christmas and Boxing Day",
    block="10_15",
)
plt.show()

## Looping and saving

In [ ]:
def batch_export_discrete_maps(
    df,
    info,
    holiday_batch=HOLIDAY_BATCH,
    fig_root=FIG_ROOT_SEMINAR,
):
    total = len(holiday_batch) * len(ALL_BLOCKS)
    done  = 0

    for group_name, holidays in holiday_batch:
        # Align holiday keys with the rename applied inside the mapping function
        holidays = [
            "Queen's Birthday" if h == "Monarch's Birthday" else h
            for h in holidays
        ]

        folder_name = group_name.replace("/", "-").replace("'", "")
        out_dir = fig_root / folder_name
        out_dir.mkdir(parents=True, exist_ok=True)

        for block in ALL_BLOCKS:
            out_path = out_dir / f"{block}.png"
            try:
                fig, _ = map_ph_we_wd_block_combined_discrete(
                    df=df,
                    info=info,
                    holidays=holidays,
                    holiday_group_name=group_name,
                    block=block,
                )
            except Exception as e:
                print(f"  [SKIP] {group_name} / {block} — {e}")
                done += 1
                continue

            fig.savefig(out_path, dpi=150, bbox_inches="tight")
            plt.close(fig)
            done += 1
            print(f"[{done}/{total}] Saved: {out_path}")

    print("\nDone.")

In [ ]:
batch_export_discrete_maps(df=rank, info=info)

In [ ]:
info.columns.tolist()

# Residual plots for seminar
- getting the difference of mean relative rank for each panel (pub hol, difference from weekend, difference from weekday)
- residuals = 3pm - 8pm MINUS 10am - 3pm mean relative demand on Queen's Birthday

The key idea is instead of computting ph_val, we_val and wd_val for a SINGLE block, I want to compute residual between two plots for each day type

In [ ]:
# 1. function signature - degining what this function is doing

def map_block_residual_three_panel( # here we are defining the function, and the parameters the function needs
    # dataframes
    df, # main dataframe
    info, # info dataframe
    # what to plot
    holidays,
    holiday_group_name,
    block_a,
    block_b,
    # anything with a '=' has a defult value (or it is a default parameter)
    # this means python will automatically use defaults for everything else, everything without a '=' is specific, it MUST be passed
    # column names in df and info
    df_station_col="Name",
    info_station_col="Name",
    lat_col="latitude",
    lon_col="longitude",
    residential_col="Residential",
    residential_threshold=0.75,
    block_suffix="_mean_relative_rank",
    # mapping config
    crs_epsg=3857, # number is standard web mercator
):
    """Compute residual values between two time blocks for each day type. Residuals will be a result from the difference of mean relative rank from Queen's Birthday demand from 3pm - 8pm MINUS 10am - 3pm."""

    # 2. set up - have to live within the same cell as the def function (def is like opening a container, everything that belongs inside must be indented)

    df = df.copy() # working in a safe copy incase I modify the original
    # renaming the monarch's birthday to Queen's Birthday in the demand df
    df["holiday_group"] = df["holiday_group"].replace( # the holiday_batch function passes Queen's Birthday as a group name
        {"Monarch's Birthday": "Queen's Birthday"} # this aligns them so filtering can find matching rows
    )

    # df contains columns nameed like '15_20_mean_relative_rank'
    # this combibes the block key with the suffic to locate the actual column name
    # so if block_a="15_20" and block_suffix="_mean_relative_rank" then col_a="15_20_mean_relative_rank"
    col_a = f"{block_a}{block_suffix}" 
    col_b = f"{block_b}{block_suffix}"
    
    # making them have human readable labels. Basically, this is saying "look up block_a in the dictionary, and if it's not found then just return block_a as is"
    # this is used for the figure title
    block_label_a = BLOCK_LABELS.get(block_a, block_a) #BLOCK_LABELS is a dictionary that maps "15_20" to "15:00-20:00"
    block_label_b = BLOCK_LABELS.get(block_b, block_b)
    block_cols = [f"{b}{block_suffix}" for b in ALL_BLOCKS]
    
    # filtering to holiday rows
    # filters the df down to ONLY rows where holiday_group columns match values in holidays
    # for example, if holidays=["Queen's Birthday"], you only keep rows for that holiday
    df_hol_full = df[df["holiday_group"].isin(holidays)].copy() # creating a new variable df_hol_full rather than is_holiday == True because later in the results loop I need two different subsets of data
    # all stations, only Queen's birthday grou prows
    # df_hol_full = rows for the holiday group (using to filter down to specific holiday days with is_holiday == True)
    # df_all = all rows for a station (used to compute weekend and weekday means
    # if i used is-Holiday == True here, i'd lose all the non-holiday rows and wouldn't be able to compute residual_we and residual_wd later
    # df_hol_full is middle step, already narrowed down the right holiday group, and contains all day types which we can split further inside the loop

    # 3. results loop - core of the function

    results = []
    for station in df[df_station_col].unique():
        df_s = df_hol_full[df_hol_full[df_station_col] == station] # df_s filtering both the station name AND the holiday group, so if we called for Punchbowl, all the rows for Punchbowl on Queen's Birthday dates across all years. this is just punchbowl rows that are ALSO in queen's bithday holiday group
        
        if df_s.empty or df_s[col_a].isna().all():
            results.append({info_station_col: station,
                "residual_ph": np.nan, "residual_we": np.nan, "residual_wd": np.nan})
            continue

        
        ph_rows = df_s[df_s["is_holiday"] == True] 
        # df_s[...} is saying "give me rows from df_s where..." 
        # df_s["is_holiday] is the condition. its looking up the is_holiday column for True.
        # first df_s = specific where to look, second is to specific the condition
        we_rows = df_s[(df_s["is_weekend"] == True) & (df_s["is_holiday"] == False)]
        wd_rows = df_s[(df_s["is_weekend"] == False) & (df_s["is_holiday"] == False)]

        # compute the actual resiuals. for each day type i need to take the mean of col_a minus the mean of col_b
        residual_ph = ph_rows[col_a].mean() - ph_rows[col_b].mean()
        residual_we = we_rows[col_a].mean() - we_rows[col_b].mean()
        residual_wd = wd_rows[col_a].mean() - wd_rows[col_b].mean()

        # no if condition required. because results.append(...) always runs for station that have data - there's no conditions to checl
        # if is only used when you need to decide between two paths
        # each time it loops through results.append(...) it adds a dictionary for that station to the list
        results.append({
            info_station_col: station,
            "residual_ph": residual_ph,
            "residual_we": residual_we,
            "residual_wd": residual_wd,
        })
        
        # so after the loop finishes, results contains one dictionrary per station, each with the station name and its three residaul values
    
    results_df = pd.DataFrame(results)

    # 4. filtering to high residential stations
    high_res = info[
        (info[residential_col] >= residential_threshold)
        & (info[info_station_col] !="Umina")
    ][info_station_col] # from the filtered rows, only give me the station name column
    # think of this part in 2 sections (info[conditions]
                                    #  info[conditions][column])
    stations_with_data = (
        df.groupby(df_station_col)[block_cols] # grouping rows of df by station name - so all punchbowl rows are grouped together etc. (.groupuby(df_station_col) indexes the station names
          .apply(lambda x: x.notna().any().any())  # runs a check on each group. notna() returns True where values are NOT null
          # its chekcing for stations that have atleast one non-null value in any block columns
          # first .any() - checks if any values in any column is not null
          # second .any() - checks if any of the columns passed that check (first .any())
        # so it returns 'True' for a station if it has any data at all in any block column, and False if it's completely empty
    )

    stations_with_data = stations_with_data[stations_with_data].index # stations_with_data[stations_with_data] filters to only the 'True' values (stations with data) and .index gives you just the station names from those rows
    # creates a list of station names with data
    # .index is extracting those names from the filtereed result
    
    eligible_stations = set(high_res).intersection(stations_with_data) # gives only the stations that appear in BOTH lists (so a station must be high residential AND have data to be included). if a station is high residential but not data, its exluded.
    
    results_df = results_df[results_df[info_station_col].isin(eligible_stations)] # results_df already exists from after the loop (its computed residuals). this line is filtering results_df down to only the eligible stations, dropping any that didn't make the cut (from filtering above)
    gdf = info[info[info_station_col].isin(eligible_stations)].copy() # gives the metadata for only the stations I want toplot
    gdf = gdf.merge(results_df, on=info_station_col, how="left") # this line is then merging computed residuals onto the metadata

    gdf = gpd.GeoDataFrame( #GeoDataFrame is converting regular dataframe (gdf) insot one that understands geomtery (point locations) so that geopandas knows where to plot each dot on the map
        gdf,
        geometry=gpd.points_from_xy(gdf[lon_col], gdf[lat_col]), # creates point geomtry from coordinate columns
        crs="EPSG:4326", #tells coordinates are in standard lat/lon
    ).to_crs(crs_epsg) # converted to web mercator so it lines up with basemap tiles

# 6. colour scale # calculating the absolute max of the data to ensure it's all being included
    step= 0.05
    abs_max = np.ceil(max(  # using np.ceil (value/step) * step divdes the step size, np.ceil rounds it to the nearest whole number and * step multiples back by the step size
    abs(results_df["residual_ph"].max()),
    abs(results_df["residual_ph"].min()),
    abs(results_df["residual_we"].max()),
    abs(results_df["residual_we"].min()),
    abs(results_df["residual_wd"].max()),
    abs(results_df["residual_wd"].min()),
    ) / step)* step

    bounds = np.round(np.arange(-abs_max, abs_max + step, step),10) #np.arange stops before the end value, so without + step, the last boundary would be missed, cutting the top off the colour bar
    # bounds is your list of bin edges

    # building colourmap
    base = matplotlib.colormaps.get_cmap("RdBu_r")
    listed = ListedColormap([base(i / (len(bounds) - 2)) for i in range(len(bounds) - 1)])
    norm = BoundaryNorm(bounds, len(bounds) -1)
    # len(bounds)-1 = if you have 15 edges, you have 14 bins between them so len(bounds) -1 gives you the number of colours needed, one per bin, it uses BoundaryNorm and range(len(bounds)-1) to generate one colour per bin
    # len(bounds)-2) = used to normalise the colour index between 0 and 1, so it spans the full colourmap. if you have 14 bins, dividing i by 12 (len(bounds)-2) nmeans
        # first bin 0/13 => start of colourmap
        # last bin 12/13 => end of colourmap
    #  dividing by len(bounds)-2 spaces the colours evenly across the full colourmap from 0.0 to 1.0, so each bin gets a unique colour thats equally spaces from its neighbours. without this, the colours wouldn't span the full range of the colurmap

    # adding extent for plotting: it defines the boundaries onf the map
    minx, miny, maxx, maxy = gdf.total_bounds # the four boundary values of your map extracted from gdf.total_bounds
    pad = 4800 # adds 4800m of breathing room around 4 edges of outermost station
    extent = (minx - pad, maxx + pad, miny - pad, maxy + pad)
    
# 7. plotting
    fig, axes = plt.subplots(1,3, figsize=(20,7))
    for ax in axes:
        ax.set_aspect("equal", adjustable="box") # keeps map proportions correct so geography isn't stretched or squished
        # adjustable="box" adjust the plot box size to maintain the equal aspect ratio rather than changing axis limits

    # finding the in and max year
    # int = integer, convets value to a whole number (without it i might get 2004.0 rather than 2004)
    year_min = int(df_hol_full["year"].min()) if not df_hol_full.empty else "?"
    year_max = int(df_hol_full["year"].max()) if not df_hol_full.empty else "?"

    fig.suptitle(
        f"{holiday_group_name}  |  {block_label_a} - {block_label_b}  ({year_min}–{year_max})",
        fontsize=12, fontweight="bold", y=1.01
    )
    axes[0].set_title(
        f"Public Holiday Temporal Shift", fontsize=12, y=0.99
    )
    axes[1].set_title(
        f"Temporal Shift Relative to Weekend Mean", fontsize=12, y=0.99
    )
    axes[2].set_title(
        f"Temporal Shift Relative to Weekday Mean", fontsize=12, y=0.99
    )

    # plotting using MARKER_KWARGS. allows me to plot the multiple axes without typing it out individually. I need to use gdf.plot() as it creates the actual dots for the map
    MARKER_KWARGS = dict(
        markersize=50,
        edgecolor="black",
        legend=False,
        missing_kwds={"color":"#e6e6e6","label":"No Data"},
    )
    gdf.plot(ax=axes[0], column="residual_ph", cmap=listed, norm=norm, **MARKER_KWARGS) # draws the dots for panel 1
    gdf.plot(ax=axes[1], column="residual_we", cmap=listed, norm=norm, **MARKER_KWARGS) # draws the dots for panel 2
    gdf.plot(ax=axes[2], column="residual_wd", cmap=listed, norm=norm, **MARKER_KWARGS) # draws the dots for panel 3

    # adding colourbars is done separately
    # sm = plt.cm.ScalarMappable(cmap=listed, norm=norm)
    # sm._A=[] # always needed when i create scalarmappable, its a workaround matplotlib quirk (matplotlib expects data to be attached to a scalarmappable)
    # cb = fig.colorbar(sm, ax=axes[0], orientation="vertical",
    #                   fraction=0.046, pad=0.02, location="left",
    #                   boundaries=bounds, ticks=bounds)
    # cb.set_label("Difference in Mean Relative Rank", fontsize=8)

    # creating a loop to plot through the other 2 axes REMOVED THE LOOP as they all have the same colourbar scale
    # for ax in axes:
    #     sm = plt.cm.ScalarMappable(cmap=listed, norm=norm)
    #     sm._A=[]
    #     cb = fig.colorbar(sm, ax=ax, orientation="vertical",
    #                   fraction=0.046, pad=0.02, location="left",
    #                   boundaries=bounds, ticks=bounds)
    #     cb.set_label("Temporal Shift in in Mean Relative Rank", fontsize=8)

    sm = plt.cm.ScalarMappable(cmap=listed, norm=norm)
    sm._A=[]
    cb = fig.colorbar(sm, ax=list(axes), orientation="horizontal",
                  fraction=0.04, pad=0.09, location="bottom",
                  aspect=40, boundaries=bounds, ticks=bounds)
    cb.set_label("Temporal Shift in in Mean Relative Rank", fontsize=10)
    
    # basemap and station labels
    for ax in axes:
        ax.set_xlim(extent[0], extent[1]) # extent defines the boundary of the map.
        ax.set_ylim(extent[2], extent[3]) #extent[0-1]: x/longitude, extent[2-3]: bottom and top (y/lat)
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
        for _, row in gdf.iterrows():
            ax.text(row.geometry.x + 800, row.geometry.y, row[info_station_col],
                    fontsize=8, ha="left", va="center")
            ax.set_axis_off()

    plt.subplots_adjust(top=0.86, bottom=0.16, wspace=0.08, hspace=0.01)
    return fig, gdf

In [ ]:
rank.columns.tolist()

In [ ]:
fig, gdf = map_block_residual_three_panel( # storing two things the function reutnrs (figure and geodataframe)
    df=rank, # passing rank df
    info=info, #passing info df
    holidays=["Queen's Birthday"], # holiday i want to plot
    holiday_group_name="Queen's Birthday", # display name for title
    block_a="15_20", # first time block
    block_b="10_15",# second time block to substract
)
plt.show() #display
fig.savefig(
    FIG_ROOT_SEMINAR / "residual_queens_birthday_3_8_10_3.png",
    dpi=150,
    bbox_inches="tight",
)